# Feature overview of the model generator 

This notebook goes over improvements made to the norfolk scripts. 

In [ ]:
from pathlib import Path 

ROOT = Path.cwd().resolve().parent.parent
SRC = ROOT / "src" 
WORK = ROOT / "work"

import sys
sys.path.append(str(SRC))

from model import norfolk_model as nm
from model import norfolk_ensemble as nensemble 
from model import pfwrite as pfw 
from model.units import ureg
import scipy.stats as stats
import numpy as np

## Unified model characterization 

Models are now defined in their own class with minimal characterization. Derived quantities are now implemented as class properties, meaning they are always up to date with any changes the user makes to the NorfolkModel instance. This reduces the headache of having to track shared quantities across multiple notebooks. 

Here, we define an example 'template' model with minimal specification outside of general grid measurements. Notice that unit types are enforced to further reduce ambiguity. 

In [ ]:
template_model = nm.NorfolkModel(model_name = 'template',
                                 lx = 1100.0 * ureg.meter,
                                 lz = 20.0 * ureg.meter,
                                 nx = 440, 
                                 nz = 200,
                                 origin = (0.0 * ureg.meter, -15.0 * ureg.meter),
                                 l_land = 1000.0 * ureg.meter,
                                 l_shelf = 100.0 * ureg.meter,
                                 ELEVATION_LEFT = 5.0 * ureg.meter,
                                 mean_sea_level_elevation = 0.0 * ureg.meter,
                                 slope_elevation = -8.0 * ureg.meter)

# Can also access a string summary of the model (WIP)
print(template_model)

Here, we make a change to the air pressure at sea level. The hydrostatic pressures on the slope regions which depend on this quantity do not have to be manually retabulated, so long as you access the quantity through the property call. Note that property calls are function calls, so if you repeatedly access a property it may make more sense to store it as a variable first. 

In [ ]:
print(template_model.SPINUP_AIR_PRESSURE_AT_SEA_LEVEL)
print(template_model.spinup_slope_pressure_profile_record[0])
template_model.SPINUP_AIR_PRESSURE_AT_SEA_LEVEL = 200000.0 * ureg.pascal
print(template_model.spinup_slope_pressure_profile_record[0])

## Electric Fencing 
The model class will attempt to catch when the user attempts to define a model that has inconsistencies. 

In [ ]:
land_profile = np.linspace(5.0, 0, 10)
try: 
    bad_model = nm.NorfolkModel(model_name = 'bad_model',
                                land_nx = 5,
                                land_profile = land_profile * ureg.meter,
    )
except Exception as e:
    print(f"Zap! I told you that {e} would happen!")                            

## Functional expressions 

The calculations have been rewritten in a functional style rather than imperative to more naturally express the assumptions of the model. 

An example below is extracting the elevations of a region and transforming those into pressure coordinates. 

In [ ]:
region = template_model.region_creek
zindex_of_region = template_model.id_to_nz(region)
elevations_of_region = template_model.nz_to_elevation(zindex_of_region)
pressures_on_region = template_model.pressure_coordinates_at(elevations_of_region, 
                                                             reference_elevation= 0.0 * ureg.meter,
                                                             reference_head = 101325.0 * ureg.pascal,
                                                             density= 1000.0 * ureg.kilogram / ureg.meter**3)

print(pressures_on_region)

# Compact distribution descriptor  

Now we define the distribution(s) over the model which our parameters will come from. In our example we mostly believe the guidance for LLNL, but maybe obtain some noisy sensor data about precipitation in the region. No problem! We can easily hot swap any distribution for another, in this case we change the default uniform distribution to a Gamma distribution with mean 2.5e-8.  

In [ ]:
example_ensemble = nensemble.NorfolkEnsemble(NorfolkModel=template_model, recharge_dist=stats.gamma(a=1, scale=2.5e-8))


Now we can draw model parameters. 

In [ ]:
my_samples = example_ensemble.draw(3)

## Separation of PFLOTRAN and Model 

The model is characterized as abstractly as possible. The user is exposed to the 'pfwrite' module which is conceptually a translation layer between a NorfolkModel and its representation in terms of PFLOTRAN inputs. This lets the user think about the details of the model without having to also think about the details of PFLOTRAN.  

Let's write our list of NorfolkModels out and track the progress. 

In [ ]:
from tqdm import tqdm

my_test_ensemble = './test_ensemble'
for realization in tqdm(my_samples, desc="Writing Norfolk Models"): 
    name = realization.model_name
    realization_dir = f'/{WORK}/{my_test_ensemble}/{name}'
    pfw.write(realization, realization_dir)

# Testing, Diffing, and Reproducibility 

Say we want to store this specific draw of NorfolkModels for later. Since it's just a list of homogeneous objects, we can pickle it into a compact binary file. This makes it easier to version control and inspect, rather than having to track 100 input files per simulation.  

In [ ]:
import pickle

with open('my_samples_v1.nfms', 'wb') as f:
    pickle.dump(my_samples, f)

After we take a look at the simulation output, maybe we like but want to tweak the recharge rate. 

------ WIP CURRENTLY ISSUES WITH UNIT REGISTRY ------

In [ ]:
with open('my_samples_v1.nfms', 'rb') as f:
    loaded_samples = pickle.load(f)

for model in loaded_samples:
    model.recharge = model.recharge + 0.1 * ureg.meter / ureg.year 

loaded_samples.name = "Diff test with 0.1 increase in recharge"

with open('my_samples_v2.nfms', 'wb') as f:
    pickle.dump(loaded_samples, f)

Easy peasy! The module also comes with a suite of unit tests (pytest) you can add to, if you want to QC what you're getting. 

# TACC-Specific stuff
On TACC, parameter sweeps are best run using a utility called Pylauncher. Pylauncher cycles through a list of run commands rather than trying to run every realization simulataneously. This is advantageous to reduce queue waiting times, because you can request a submission size relative to the amount of nodes are currently idling (which you can live query on the TACC portal). In addition, the queuestate is cached so you can continue a batch job if it's interrupted without starting the batch over again. 

The below will write the batch job file for input to pylauncher. Note that this submits *both* the spinup and post-spinup run. The pfwrite module inserts a dangling symlink to the spinup run allowing the simulation to start without user intervention. Neat! 

In [ ]:
with open(f'{WORK}/{my_test_ensemble}/parallellines', 'w') as batchfile:
    for realization in my_samples: 
        name = realization.model_name
        pylaunch_cmd_spinup = 'PYL_MPIEXEC $PFLOW_DIR/pflotran/pflotran -input_prefix pflotran_spinup'
        pylaunch_cmd_post = 'PYL_MPIEXEC $PFLOW_DIR/pflotran/pflotran -input_prefix pflotran'
        batchfile.write(f'cd {name}/spinup && {pylaunch_cmd_spinup} && cd ../post && {pylaunch_cmd_post}\n')
        

## Installing PFLOTRAN on TACC Systems

You can install PFLOTRAN completely from source including its dependencies, but you can also use TACC's precompiled modules to link into PFLOTRAN. However, this does not work out of the box and requires some jerry-rigging. Instructions on the CHG wiki page [here](https://cloud.wikis.utexas.edu/wiki/external/N2JkMDMzY2UxNDMxNDZiNGJkMjRiNDJmZTJmYWVjZDc).

Really, the question is which kind of headache do you want. 🫡